# 📊 Bank Account Fraud - Notebook 01: EDA & Data Cleaning
Dự án phân tích bộ dữ liệu **NeurIPS 2022 Bank Account Fraud (BAF)**.

### Mục tiêu của Notebook:
1. Khám phá tổng quan tập dữ liệu 1,000,000 dòng (`data/raw/Base.csv`).
2. Phát hiện và xử lý các giá trị Sentinel (`-1`) đại diện cho missing values.
3. Tối ưu hóa bộ nhớ (RAM Downcasting) và trích xuất đặc trưng mới.
4. Lưu trữ dữ liệu sạch vào `data/processed/clean_base.parquet`.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns

# Thêm root path để import từ src
sys.path.append("..")
from src.data.load_data import load_raw_dataset, query_duckdb
from src.features.clean_and_engineer import clean_and_transform_pipeline

print("Thư viện đã sẵn sàng!")

## 1. Nhanh chóng thống kê dữ liệu thô với DuckDB

In [ ]:
conn = duckdb.connect()
query_summary = """
SELECT 
    COUNT(*) AS total_rows,
    SUM(fraud_bool) AS fraud_count,
    ROUND(AVG(fraud_bool) * 100, 3) AS fraud_rate_pct,
    COUNT(DISTINCT month) AS num_months
FROM "../data/raw/Base.csv"
"""
conn.execute(query_summary).df()

## 2. Kiểm tra các giá trị Sentinel (-1) trong tập dữ liệu
Trong bộ dữ liệu BAF NeurIPS 2022, nhiều thuộc tính mã hóa dữ liệu thiếu bằng `-1`:

In [ ]:
missing_stats_sql = """
SELECT 
    COUNT(*) AS total,
    ROUND(COUNT(CASE WHEN prev_address_months_count = -1 THEN 1 END) * 100.0 / COUNT(*), 2) AS prev_address_missing_pct,
    ROUND(COUNT(CASE WHEN intended_balcon_amount < 0 THEN 1 END) * 100.0 / COUNT(*), 2) AS intended_balcon_missing_pct,
    ROUND(COUNT(CASE WHEN bank_months_count = -1 THEN 1 END) * 100.0 / COUNT(*), 2) AS bank_months_missing_pct,
    ROUND(COUNT(CASE WHEN session_length_in_minutes < 0 THEN 1 END) * 100.0 / COUNT(*), 2) AS session_length_missing_pct
FROM "../data/raw/Base.csv"
"""
conn.execute(missing_stats_sql).df()

## 3. Thực thi Pipeline Làm Sạch & Kỹ Thuật Đặc Trưng (Feature Engineering)

In [ ]:
raw_df = load_raw_dataset("Base.csv")
clean_df = clean_and_transform_pipeline(raw_df)
clean_df.head()

## 4. Lưu dữ liệu đã làm sạch vào định dạng Parquet tối ưu

In [ ]:
out_path = Path("../data/processed/clean_base.parquet")
clean_df.to_parquet(out_path, index=False, engine="pyarrow")
print(f"Đã lưu: {out_path} ({out_path.stat().st_size / (1024**2):.2f} MB)")